# 06 - Train, tune, and evaluate

**Job:** establish a simple baseline, tune a class-weighted logistic
regression on validation data, choose a validation threshold, and
touch the test set once at the end.

**Primary metric:** average precision (PR-AUC), because late deliveries
are the minority. Late-class precision, recall, F1, ROC-AUC, balanced
accuracy, and accuracy are supporting metrics.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score,
)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
ARTIFACT_DIR = ROOT / "artifacts"

# Training and validation only. Test is intentionally not loaded here.
X_train = sparse.load_npz(ARTIFACT_DIR / "05_X_train.npz")
y_train = np.load(ARTIFACT_DIR / "05_y_train.npy")
X_validation = sparse.load_npz(ARTIFACT_DIR / "05_X_validation.npz")
y_validation = np.load(ARTIFACT_DIR / "05_y_validation.npy")
feature_names = json.loads((ARTIFACT_DIR / "05_feature_list.json").read_text(encoding="utf-8"))
print(f"Train: {X_train.shape}; validation: {X_validation.shape}; features: {len(feature_names)}")

Train: (67529, 142); validation: (14470, 142); features: 142


In [2]:
def metric_row(y_true, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype("int8")
    return {
        "average_precision": float(average_precision_score(y_true, probabilities)),
        "roc_auc": float(roc_auc_score(y_true, probabilities)),
        "precision_late": float(precision_score(y_true, predictions, zero_division=0)),
        "recall_late": float(recall_score(y_true, predictions, zero_division=0)),
        "f1_late": float(f1_score(y_true, predictions, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predictions)),
        "accuracy": float(accuracy_score(y_true, predictions)),
        "threshold": float(threshold),
    }

## Baseline

The prior baseline ignores all features and predicts the training
class probability. It defines the minimum result a useful model must
beat; its raw accuracy can look high while minority recall is zero.

In [3]:
baseline = DummyClassifier(strategy="prior")
baseline.fit(X_train, y_train)
baseline_validation_prob = baseline.predict_proba(X_validation)[:, 1]
baseline_validation = metric_row(y_validation, baseline_validation_prob, threshold=0.5)
print(pd.Series(baseline_validation, name="validation_prior_baseline"))

average_precision    0.053421
roc_auc              0.500000
precision_late       0.000000
recall_late          0.000000
f1_late              0.000000
balanced_accuracy    0.500000
accuracy             0.946579
threshold            0.500000
Name: validation_prior_baseline, dtype: float64


## Tune model strength on validation PR-AUC

In [4]:
tuning_records = []
fitted_candidates = {}
for C in [0.01, 0.1, 1.0, 10.0]:
    candidate = LogisticRegression(
        C=C,
        class_weight="balanced",
        solver="liblinear",
        max_iter=1000,
        random_state=42,
    )
    candidate.fit(X_train, y_train)
    probabilities = candidate.predict_proba(X_validation)[:, 1]
    metrics = metric_row(y_validation, probabilities, threshold=0.5)
    tuning_records.append({"C": C, **metrics})
    fitted_candidates[C] = (candidate, probabilities)

tuning = pd.DataFrame(tuning_records).sort_values(
    ["average_precision", "f1_late"], ascending=False
).reset_index(drop=True)
display(tuning)
best_C = float(tuning.iloc[0]["C"])
selected_model, selected_validation_prob = fitted_candidates[best_C]
print(f"Selected C={best_C:g} by validation average precision.")

,C,average_precision,roc_auc,precision_late,recall_late,f1_late,balanced_accuracy,accuracy,threshold
0,0.01,0.143898,0.757194,0.070077,0.967658,0.130689,0.621487,0.312301,0.5
1,10.00,0.143329,0.748476,0.070573,0.945666,0.131345,0.621406,0.331790,0.5
2,0.10,0.143189,0.752783,0.070477,0.950841,0.131227,0.621547,0.327436,0.5
3,1.00,0.142859,0.749166,0.070602,0.946960,0.131407,0.621724,0.331237,0.5


Selected C=0.01 by validation average precision.


## Tune the classification threshold on validation late-class F1

In [5]:
threshold_rows = []
for threshold in np.linspace(0.05, 0.95, 91):
    row = metric_row(y_validation, selected_validation_prob, threshold=float(threshold))
    threshold_rows.append(row)
threshold_results = pd.DataFrame(threshold_rows).sort_values(
    ["f1_late", "recall_late", "precision_late"], ascending=False
).reset_index(drop=True)
best_threshold = float(threshold_results.iloc[0]["threshold"])
selected_validation = metric_row(y_validation, selected_validation_prob, best_threshold)
display(threshold_results.head(10))
print(f"Selected threshold={best_threshold:.2f} by validation late-class F1.")

tuning.to_csv(ARTIFACT_DIR / "06_validation_tuning.csv", index=False)
threshold_results.to_csv(ARTIFACT_DIR / "06_threshold_tuning.csv", index=False)

,average_precision,roc_auc,precision_late,recall_late,f1_late,balanced_accuracy,accuracy,threshold
0,0.143898,0.757194,0.184727,0.328590,0.236499,0.623374,0.886662,0.79
1,0.143898,0.757194,0.192403,0.301423,0.234879,0.615010,0.895093,0.80
2,0.143898,0.757194,0.172879,0.347995,0.231000,0.627016,0.876227,0.78
3,0.143898,0.757194,0.164273,0.369987,0.227526,0.631880,0.865791,0.77
4,0.143898,0.757194,0.156153,0.390686,0.223125,0.635768,0.854665,0.76
5,0.143898,0.757194,0.192085,0.257439,0.220011,0.598165,0.902488,0.81
6,0.143898,0.757194,0.149323,0.413972,0.219479,0.640438,0.842709,0.75
7,0.143898,0.757194,0.139669,0.469599,0.215302,0.653176,0.817139,0.73
8,0.143898,0.757194,0.142676,0.437257,0.215150,0.644488,0.829578,0.74
9,0.143898,0.757194,0.135257,0.503234,0.213209,0.660831,0.801589,0.72


Selected threshold=0.79 by validation late-class F1.


## Final fit and one-time test evaluation

Hyperparameters and threshold are now frozen. The classifier is
refit on train + validation transformed features. The test matrices
and labels are loaded for the first and only time below.

In [6]:
X_train_validation = sparse.vstack([X_train, X_validation], format="csr")
y_train_validation = np.concatenate([y_train, y_validation])

final_model = LogisticRegression(
    C=best_C,
    class_weight="balanced",
    solver="liblinear",
    max_iter=1000,
    random_state=42,
)
final_model.fit(X_train_validation, y_train_validation)
final_baseline = DummyClassifier(strategy="prior").fit(X_train_validation, y_train_validation)

# TEST SET TOUCH: exactly once, after every choice is frozen.
X_test = sparse.load_npz(ARTIFACT_DIR / "05_X_test.npz")
y_test = np.load(ARTIFACT_DIR / "05_y_test.npy")
test_order_ids = pd.read_csv(ARTIFACT_DIR / "05_order_ids_test.csv.gz")["order_id"]

baseline_test_prob = final_baseline.predict_proba(X_test)[:, 1]
model_test_prob = final_model.predict_proba(X_test)[:, 1]
baseline_test = metric_row(y_test, baseline_test_prob, threshold=0.5)
model_test = metric_row(y_test, model_test_prob, threshold=best_threshold)

results = pd.DataFrame([
    {"split": "validation", "model": "prior_baseline", **baseline_validation},
    {"split": "validation", "model": "logistic_regression", **selected_validation},
    {"split": "test", "model": "prior_baseline", **baseline_test},
    {"split": "test", "model": "logistic_regression", **model_test},
])
display(results)

assert model_test["average_precision"] > baseline_test["average_precision"], (
    "The trained model did not beat baseline test PR-AUC."
)

,split,model,average_precision,roc_auc,precision_late,recall_late,f1_late,balanced_accuracy,accuracy,threshold
0,validation,prior_baseline,0.053421,0.500000,0.000000,0.000000,0.000000,0.500000,0.946579,0.50
1,validation,logistic_regression,0.143898,0.757194,0.184727,0.328590,0.236499,0.623374,0.886662,0.79
2,test,prior_baseline,0.066132,0.500000,0.000000,0.000000,0.000000,0.500000,0.933868,0.50
3,test,logistic_regression,0.124853,0.692066,0.106404,0.225705,0.144627,0.545737,0.823440,0.79


## Save the trained model and results summary

In [7]:
bundle = {
    "classifier": final_model,
    "decision_threshold": best_threshold,
    "selected_C": best_C,
    "feature_names": feature_names,
    "primary_metric": "average_precision",
    "positive_class": "late",
    "test_evaluated_once": True,
}
joblib.dump(bundle, ARTIFACT_DIR / "06_final_model.joblib")
results.to_csv(ARTIFACT_DIR / "06_results_summary.csv", index=False)

predictions = pd.DataFrame({
    "order_id": test_order_ids,
    "actual_late": y_test,
    "predicted_late_probability": model_test_prob,
    "predicted_late": (model_test_prob >= best_threshold).astype("int8"),
})
predictions.to_csv(
    ARTIFACT_DIR / "06_test_predictions.csv.gz", index=False, compression="gzip"
)

result_payload = {
    "primary_metric": "average_precision",
    "selected_C": best_C,
    "selected_threshold": best_threshold,
    "validation": {
        "baseline": baseline_validation,
        "logistic_regression": selected_validation,
    },
    "test": {
        "baseline": baseline_test,
        "logistic_regression": model_test,
    },
    "test_evaluated_once": True,
}
(ARTIFACT_DIR / "06_results_summary.json").write_text(
    json.dumps(result_payload, indent=2), encoding="utf-8"
)

summary_md = f'''# Model results summary

## Selection

- Primary metric: average precision (PR-AUC)
- Selected logistic-regression C: `{best_C:g}`
- Selected validation F1 threshold: `{best_threshold:.2f}`
- Preprocessing was fit on training only; test was evaluated once after choices were frozen.

## Test comparison

| Model | PR-AUC | ROC-AUC | Late precision | Late recall | Late F1 | Balanced accuracy | Accuracy |
|---|---:|---:|---:|---:|---:|---:|---:|
| Prior baseline | {baseline_test['average_precision']:.3f} | {baseline_test['roc_auc']:.3f} | {baseline_test['precision_late']:.3f} | {baseline_test['recall_late']:.3f} | {baseline_test['f1_late']:.3f} | {baseline_test['balanced_accuracy']:.3f} | {baseline_test['accuracy']:.3f} |
| Logistic regression | {model_test['average_precision']:.3f} | {model_test['roc_auc']:.3f} | {model_test['precision_late']:.3f} | {model_test['recall_late']:.3f} | {model_test['f1_late']:.3f} | {model_test['balanced_accuracy']:.3f} | {model_test['accuracy']:.3f} |

The trained model beats the feature-free baseline on test PR-AUC. Accuracy is included for context but is not used to select the model because the target is imbalanced.
'''
(ARTIFACT_DIR / "06_results_summary.md").write_text(summary_md, encoding="utf-8")
print(summary_md)

reloaded_bundle = joblib.load(ARTIFACT_DIR / "06_final_model.joblib")
assert reloaded_bundle["decision_threshold"] == best_threshold
assert reloaded_bundle["classifier"].n_features_in_ == len(feature_names)
print("Saved and reloaded the final model bundle successfully.")

# Model results summary

## Selection

- Primary metric: average precision (PR-AUC)
- Selected logistic-regression C: `0.01`
- Selected validation F1 threshold: `0.79`
- Preprocessing was fit on training only; test was evaluated once after choices were frozen.

## Test comparison

| Model | PR-AUC | ROC-AUC | Late precision | Late recall | Late F1 | Balanced accuracy | Accuracy |
|---|---:|---:|---:|---:|---:|---:|---:|
| Prior baseline | 0.066 | 0.500 | 0.000 | 0.000 | 0.000 | 0.500 | 0.934 |
| Logistic regression | 0.125 | 0.692 | 0.106 | 0.226 | 0.145 | 0.546 | 0.823 |

The trained model beats the feature-free baseline on test PR-AUC. Accuracy is included for context but is not used to select the model because the target is imbalanced.

Saved and reloaded the final model bundle successfully.
